In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import os

# Global style
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.titleweight": "bold",
    "font.family": "sans-serif",
})

# Brand colors (use same in Power BI)
NAVY   = "#1F3B73"
ORANGE = "#F2A93B"
GRAY   = "#6B7280"
RED    = "#C0392B"

processed = "../data/processed/"
shots = "../reports/screenshots/"
os.makedirs(shots, exist_ok=True)

print("Environment ready.")

Environment ready.


In [3]:
df = pd.read_csv(processed + "order_master.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

print("Shape:", df.shape)
print("Date range:", df["order_purchase_timestamp"].min(), "→", df["order_purchase_timestamp"].max())
print("Unique orders:", df["order_id"].nunique())
print("Unique customers:", df["customer_unique_id"].nunique())

Shape: (110197, 24)
Date range: 2016-09-15 12:16:38 → 2018-08-29 15:00:37
Unique orders: 96478
Unique customers: 93358


In [5]:
total_revenue  = df["revenue"].sum()
total_orders   = df["order_id"].nunique()
total_customers = df["customer_unique_id"].nunique()
aov            = total_revenue / total_orders

# Repeat purchase rate: % of customers with 2+ orders
orders_per_cust = df.groupby("customer_unique_id")["order_id"].nunique()
repeat_customers = (orders_per_cust > 1).sum()
repeat_rate     = repeat_customers / total_customers * 100

# Items per order (avg)
items_per_order = len(df) / total_orders

# Avg review
avg_review = df["review_score"].mean()

# Late delivery
late_rate = df["is_late"].mean() * 100

kpi = pd.DataFrame({
    "Metric": [
        "Total Revenue (BRL)",
        "Total Orders",
        "Total Customers",
        "Average Order Value (BRL)",
        "Repeat Customers",
        "Repeat Purchase Rate (%)",
        "Items per Order",
        "Avg Review Score",
        "Late Delivery Rate (%)",
    ],
    "Value": [
        f"R$ {total_revenue:,.2f}",
        f"{total_orders:,}",
        f"{total_customers:,}",
        f"R$ {aov:,.2f}",
        f"{repeat_customers:,}",
        f"{repeat_rate:.2f}%",
        f"{items_per_order:.2f}",
        f"{avg_review:.3f}",
        f"{late_rate:.2f}%",
    ]
})
print(kpi.to_string(index=False))

                   Metric            Value
      Total Revenue (BRL) R$ 15,419,773.75
             Total Orders           96,478
          Total Customers           93,358
Average Order Value (BRL)        R$ 159.83
         Repeat Customers            2,801
 Repeat Purchase Rate (%)            3.00%
          Items per Order             1.14
         Avg Review Score            4.082
   Late Delivery Rate (%)            7.91%


In [6]:
kpi.to_csv(processed + "kpi_summary.csv", index=False)
print("Saved kpi_summary.csv")

Saved kpi_summary.csv
